# Phase 5: Fairness Check on Age

We are running this analysis because of a specific empirical finding rather than a simple compliance exercise. Age emerged as the third most important feature globally in the Phase 4 SHAP evaluation. Because age is a protected characteristic under the UK Equality Act, this phase tests directly whether the model treats age groups differently.

**A Note on Tooling**
The core fairness computations below are built directly using pandas and sklearn to guarantee fully tested and verified results. An optional Fairlearn verification cell is included afterwards. This should reproduce the exact same numbers and serves as a valuable addition to your technical portfolio. If the Fairlearn library encounters environment issues, rely entirely on the manual pandas results above it. Nothing downstream depends on the Fairlearn cell executing perfectly.

**Core Fairness Metrics**
We evaluate the following metrics to understand how the model behaves across different demographic segments:
* **Selection rate:** The fraction of each age group flagged as high risk. A large gap here relates to the concept of demographic parity.
* **False positive rate:** Out of the people who would not actually default, this is the fraction the model wrongly flags per group. A gap here means the reliable customers of one group are being penalised more heavily than those of another.
* **False negative rate:** Out of the people who would actually default, this is the fraction the model misses per group. A gap here indicates that the true risk of one group is being underestimated.

It is important to understand that satisfying one of these metrics does not guarantee the others. A model can achieve equal selection rates across groups while still producing vastly different error rates. This represents a fundamental tension in fairness research rather than a flaw in our specific methodological approach.

In [13]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/cleaned_features.csv', index_col=0)
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
final_model = joblib.load('../data/processed/credit_risk_model.joblib')
print("Model and split reloaded. X_test shape:", X_test.shape)


Model and split reloaded. X_test shape: (30000, 16)


## Binning Age into Groups

Group fairness metrics require discrete categories to function properly. We therefore convert the continuous age variable into distinct bands. The brackets for ages 18 to 30 and 61 and older represent the populations most relevant to age discrimination concerns in credit lending. The middle cohorts, covering ages 31 to 45 and 46 to 60, serve as our standard baseline for comparison.

In [14]:
bins = [17, 30, 45, 60, 120]
labels = ['18-30', '31-45', '46-60', '61+']
age_group = pd.cut(X_test['age'], bins=bins, labels=labels)

print(age_group.value_counts())


age
46-60    10638
61+       8991
31-45     8202
18-30     2169
Name: count, dtype: int64


## Computing Fairness Metrics per Age Group

We evaluate the model at its standard 0.5 decision boundary to calculate three vital metrics for each age band.

By taking the mean of the model predictions within a specific group, we determine its selection rate. This reveals the overall proportion of that demographic flagged as high risk.

When we restrict our view strictly to the reliable borrowers, the mean of the predictions gives us the false positive rate. This crucial metric shows exactly how often creditworthy individuals in that age bracket are unfairly penalised.

Finally, we isolate the genuine defaulters. Taking the mean of the inverse predictions for this subset yields the false negative rate. This exposes how often the model completely fails to catch actual risk within the demographic.

In [15]:
y_pred = final_model.predict(X_test)

results = pd.DataFrame({
    'age_group': age_group.values,
    'y_true': y_test.values,
    'y_pred': y_pred
})

def group_metrics(g):
    selection_rate = g['y_pred'].mean()
    negatives = g[g['y_true'] == 0]
    positives = g[g['y_true'] == 1]
    fpr = negatives['y_pred'].mean() if len(negatives) > 0 else np.nan
    fnr = (1 - positives['y_pred']).mean() if len(positives) > 0 else np.nan
    return pd.Series({
        'n': len(g),
        'selection_rate': selection_rate,
        'false_positive_rate': fpr,
        'false_negative_rate': fnr
    })

metrics_by_group = results.groupby('age_group', observed=True).apply(group_metrics, include_groups=False)
metrics_by_group


,n,selection_rate,false_positive_rate,false_negative_rate
age_group,,,,
18-30,2169.0,0.426003,0.362342,0.131868
31-45,8202.0,0.348330,0.298789,0.174870
46-60,10638.0,0.234349,0.196997,0.246499
61+,8991.0,0.083417,0.069525,0.422764


## Evaluating the Disparities Directly

The gap between the highest and lowest group for each metric serves as the headline figure for this analysis. There is no universal legal threshold that automatically defines an unacceptable margin. However, any large and consistent gap must be taken seriously and reported honestly, regardless of the final outcome. This holds particularly true for disparities in the false positive and false negative rates, which often reveal far more about underlying model bias than the baseline selection rate alone.

In [16]:
print("Disparity (max - min across age groups):\n")
for col in ['selection_rate', 'false_positive_rate', 'false_negative_rate']:
    gap = metrics_by_group[col].max() - metrics_by_group[col].min()
    print(f"  {col}: {gap:.3f}")


Disparity (max - min across age groups):

  selection_rate: 0.343
  false_positive_rate: 0.293
  false_negative_rate: 0.291


## Optional Cross Check with Fairlearn

This step aims to reproduce the exact same selection rates calculated above using the dedicated Fairlearn library. You must run `pip install fairlearn` in a terminal window before executing this cell. If the code still produces an error, you can safely skip it and rely entirely on the manual results generated in the previous section. The core fairness audit is fully self contained and does not depend on this external library functioning perfectly.

In [17]:
try:
    from fairlearn.metrics import MetricFrame, selection_rate
    mf = MetricFrame(
        metrics=selection_rate,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=age_group
    )
    print("Fairlearn selection rate by group:")
    print(mf.by_group)
    print("\nFairlearn agrees with the manual computation above." )
except Exception as e:
    print("Fairlearn cross-check didn't run - that's fine, the manual results above stand on their own.")
    print("Error was:", e)


Fairlearn selection rate by group:
age
18-30    0.426003
31-45    0.348330
46-60    0.234349
61+      0.083417
Name: selection_rate, dtype: float64

Fairlearn agrees with the manual computation above.


## Documenting Findings and Preparing for the Final Phase

The exact disparity numbers for all three metrics require careful assessment against their respective sample sizes. A large mathematical gap within a very small group often reflects statistical noise rather than systemic bias. If the data shows no meaningful disparity, the model demonstrates baseline fairness across age groups. Conversely, if a disparity is confirmed, standard industry mitigations include removing the offending feature, reweighting the training data, or applying fairness constraints during model training.

Phase 6 transitions the project into an interactive Streamlit application. This will deliver a dual perspective dashboard featuring an internal banking interface to display these exact fairness metrics alongside the global SHAP plots, and a consumer facing interface providing the actionable recourse checklist developed in Phase 4.

# Phase 5, Part B: Is this driven by age directly, or by a proxy?

**Empirical Findings on Age Disparity**
The initial fairness audit revealed substantial disparities across age groups. The selection rate ranged from 42.6 percent for the 18 to 30 cohort down to 8.3 percent for those aged 61 and over. Similarly, the false positive rate dropped from 36.2 percent to 7.0 percent, while the false negative rate increased from 13.2 percent to 42.3 percent. Because all four age groups contain thousands of members, these figures represent a genuine systemic pattern rather than statistical noise.

**Testing for Direct Influence Versus Proxy Variables**
The analysis must now determine whether this disparity stems from the model using age directly or from other features acting as proxies, such as credit history length or utilisation patterns. To test this, the exact same model architecture is refitted with the age feature completely removed from the training data. The three fairness metrics are then recomputed on the exact same demographic groups to allow for a direct comparison.

**Interpreting the Outcomes**
Both potential outcomes provide critical insights for model governance:
* If the disparity shrinks significantly, it confirms age itself was a major direct driver of the skewed predictions.
* If the disparity barely moves, it indicates other features are acting as a proxy for age. This is arguably the more critical finding, as it proves that simply removing the protected characteristic is insufficient to resolve the underlying bias.

**Evaluating the Accuracy Trade Off**
Overall model performance metrics, specifically ROC-AUC and PR-AUC, are evaluated both with and without the age feature. Removing a highly predictive variable to address fairness concerns often incurs a tangible cost to overall accuracy. Quantifying this exact trade off provides the necessary context for stakeholders and model governance committees to make informed deployment decisions.

In [18]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Baseline (with age) - reuse the model and predictions from earlier in this notebook
y_pred_with_age = final_model.predict(X_test)
results_with_age = pd.DataFrame({
    'age_group': age_group.values, 'y_true': y_test.values, 'y_pred': y_pred_with_age
})
metrics_with_age = results_with_age.groupby('age_group', observed=True).apply(group_metrics, include_groups=False)

probs_with_age = final_model.predict_proba(X_test)[:, 1]
auc_with_age = roc_auc_score(y_test, probs_with_age)
pr_auc_with_age = average_precision_score(y_test, probs_with_age)

print("Baseline (with age) - ROC-AUC:", round(auc_with_age, 3), "PR-AUC:", round(pr_auc_with_age, 3))


Baseline (with age) - ROC-AUC: 0.869 PR-AUC: 0.403


## Refitting the Model Without Age

The age feature is dropped entirely from both the training and test datasets. The model never sees this variable during training or prediction. The original `age_group` labels, which were computed before dropping the column, are retained purely to evaluate the outcome by demographic group. The model itself operates with absolutely no access to this demographic information.

In [19]:
X_train_no_age = X_train.drop(columns=['age'])
X_test_no_age = X_test.drop(columns=['age'])

model_no_age = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
model_no_age.fit(X_train_no_age, y_train)

y_pred_no_age = model_no_age.predict(X_test_no_age)
probs_no_age = model_no_age.predict_proba(X_test_no_age)[:, 1]

auc_no_age = roc_auc_score(y_test, probs_no_age)
pr_auc_no_age = average_precision_score(y_test, probs_no_age)

print("Without age - ROC-AUC:", round(auc_no_age, 3), "PR-AUC:", round(pr_auc_no_age, 3))
print(f"\nPerformance change: ROC-AUC {auc_no_age - auc_with_age:+.3f}, PR-AUC {pr_auc_no_age - pr_auc_with_age:+.3f}")


Without age - ROC-AUC: 0.866 PR-AUC: 0.4

Performance change: ROC-AUC -0.004, PR-AUC -0.003


## Recompute the disparity without age, and compare directly

In [20]:
results_no_age = pd.DataFrame({
    'age_group': age_group.values, 'y_true': y_test.values, 'y_pred': y_pred_no_age
})
metrics_no_age = results_no_age.groupby('age_group', observed=True).apply(group_metrics, include_groups=False)

print("WITH age:\n", metrics_with_age)
print("\nWITHOUT age:\n", metrics_no_age)

print("\nDisparity comparison (max - min across age groups):")
for col in ['selection_rate', 'false_positive_rate', 'false_negative_rate']:
    gap_with = metrics_with_age[col].max() - metrics_with_age[col].min()
    gap_without = metrics_no_age[col].max() - metrics_no_age[col].min()
    print(f"  {col}: with age = {gap_with:.3f}  |  without age = {gap_without:.3f}")


WITH age:
                  n  selection_rate  false_positive_rate  false_negative_rate
age_group                                                                   
18-30       2169.0        0.426003             0.362342             0.131868
31-45       8202.0        0.348330             0.298789             0.174870
46-60      10638.0        0.234349             0.196997             0.246499
61+         8991.0        0.083417             0.069525             0.422764

WITHOUT age:
                  n  selection_rate  false_positive_rate  false_negative_rate
age_group                                                                   
18-30       2169.0        0.397418             0.333333             0.157509
31-45       8202.0        0.310168             0.259354             0.200777
46-60      10638.0        0.235759             0.198307             0.243697
61+         8991.0        0.130909             0.116181             0.345528

Disparity comparison (max - min across age groups

## Documenting the Fairness and Accuracy Trade Off

The final step in this phase evaluates the disparity metrics alongside the changes in overall model performance, specifically looking at the ROC AUC and PR AUC scores after removing the age feature. The outcome of this comparison provides a definitive conclusion regarding model bias.

If the demographic disparity shrinks significantly, it proves age was a direct driver of the skewed predictions. In this scenario, removing the feature reduces the gap, though this must be weighed against the quantified cost to predictive accuracy. Conversely, if the disparity remains largely unchanged, it demonstrates that other features are acting as a proxy for age. This confirms that simply omitting a protected characteristic from the training data is not a sufficient solution on its own. 

The project now moves into Phase 6 to construct the interactive Streamlit application. The internal banking interface will prominently feature this comprehensive fairness analysis. This ensures stakeholders can review not just the baseline demographic disparity, but the results of a robust empirical test determining whether that bias is direct or driven by proxy variables.


# Phase 5, Part C: Final Decision on Age Exclusion and Monitoring

**The Decision and Rationale**
The age feature is officially excluded as a model input moving forward. This aligns directly with standard industry practice. Major credit scoring models, such as FICO, explicitly avoid using age as a variable. Instead, they rely on metrics like the length of credit history. While credit history length correlates with age, it evaluates risk based on actual account behaviour rather than raw demographics. Our empirical results mirror this reality perfectly. Removing age from the training data cost a mere 0.004 in ROC AUC. This confirms that other existing features, including total past delinquencies, revolving utilisation, and credit line history, were already capturing the necessary predictive signal.

**Acknowledging the Remaining Bias**
It is crucial to state clearly that removing this feature does not entirely solve the fairness issue. The analysis in Part B demonstrated that 65 to 78 percent of the demographic disparity survives the removal of the age variable. This residual bias is carried by proxy features that have not been individually isolated. For this reason, the age column is retained within the broader dataset. It is strictly excluded from the model's decision making process, but it remains essential for ongoing disparity monitoring.

**Finalising the Project State**
This architectural choice represents the final saved state of the machine learning pipeline. The model saved from this point forward, which operates completely without the age input, is the exact version the Phase 6 dashboard will load and deploy.

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier
import joblib

X_train_final = X_train.drop(columns=['age'])
X_test_final = X_test.drop(columns=['age'])

final_model_no_age = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
final_model_no_age.fit(X_train_final, y_train)

joblib.dump(final_model_no_age, '../data/processed/credit_risk_model.joblib')
print("Saved as the canonical model - age excluded from training.")
print("Model's actual features:", list(final_model_no_age.feature_names_in_))


Saved as the canonical model - age excluded from training.
Model's actual features: ['RevolvingUtilizationOfUnsecuredLines', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'income_missing', 'dependents_missing', 'has_delinquency_data_error', 'estimated_monthly_debt_payment', 'disposable_income_estimate', 'total_past_delinquencies']


## Finalising the Fairness Audit and Preparing for Phase 6

The production model deliberately excludes age as a predictive input. This architectural choice is supported by established industry precedent, such as standard FICO scoring methods, and is validated by our own empirical testing, which demonstrated a negligible drop in predictive accuracy when the feature was removed.

It is equally important to acknowledge that this exclusion does not completely resolve the underlying demographic disparity. Because proxy variables continue to carry the majority of the bias, this residual disparity remains explicitly flagged as a critical area for ongoing model governance and refinement.

As the project transitions into Phase 6, the interactive dashboard will deploy this exact model architecture. The internal banking interface will present the complete fairness narrative. By detailing the initial disparity, the proxy variable test, and the final exclusion decision, the application provides stakeholders with a fully transparent view of the model governance process rather than simply presenting a static global importance chart.